# Lesson 12 - 에이전트 스크래치패드를 활용한 채팅 기록 축소

이 노트북은 Microsoft Agent Framework를 사용하여 긴 대화에서 컨텍스트를 관리하는 방법을 보여줍니다. 대화가 길어질수록 토큰 수가 증가하여 결국 모델의 컨텍스트 창을 초과하게 됩니다. 이를 해결하기 위해 **컨텍스트 요약 패턴**과 **에이전트 스크래치패드**를 사용하여 지속적인 메모리를 구현합니다.

## 학습 내용:
1. **컨텍스트 관리가 중요한 이유**: 토큰 제한 및 컨텍스트 창 이해하기
2. **컨텍스트 인지 에이전트**: 자신의 대화 컨텍스트를 관리하는 에이전트 구축
3. **컨텍스트 요약 패턴**: 도구를 사용하여 대화 기록 요약하기
4. **에이전트 스크래치패드**: 컨텍스트 축소 후에도 유지되는 지속적인 메모리

## 사전 준비 사항:
- 환경 변수가 설정된 Azure OpenAI 구성
- 이전 강의에서 기본 에이전트 개념 이해


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 컨텍스트 관리가 중요한 이유

모든 LLM은 유한한 **컨텍스트 창**을 가지고 있습니다 — 이는 한 번의 요청에서 처리할 수 있는 최대 토큰 수입니다. 다중 턴 대화가 진행됨에 따라:

- **토큰 수는 사용자 메시지와 어시스턴트 응답마다 선형적으로 증가**합니다.
- **프롬프트 토큰이 비용을 주도**하는데, 이는 전체 대화 기록이 매 턴 재전송되기 때문입니다.
- 결국 대화가 **컨텍스트 창을 초과**하게 되면 모델은 잘라내거나 오류를 발생시킵니다.

### 컨텍스트 관리 전략

| 전략 | 작동 방식 | 단점 |
|---|---|---|
| **잘라내기(Truncation)** | 가장 오래된 메시지를 삭제 | 초기 컨텍스트 손실 |
| **요약(Summarization)** | 오래된 메시지를 요약으로 압축 | 일부 세부 정보 손실, 하지만 핵심 내용 유지 |
| **스크래치패드 / 외부 메모리** | 대화 외부에 핵심 사실 저장 | 도구 호출 필요, 그러나 어떤 축소에도 견딤 |

이 노트북에서는 **요약**과 **스크래치패드 도구**를 결합하여 대화 기록이 압축되더라도 에이전트가 연속성을 유지할 수 있도록 합니다.


## 컨텍스트 인식 에이전트 만들기


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 긴 대화 시뮬레이션

컨텍스트가 어떻게 누적되는지 보기 위해 다중 대화를 진행해 보겠습니다. 에이전트는 대화가 진행되는 동안 주요 세부 사항(선호도, 예산, 여행 날짜)을 유지하고 연속성을 보여야 합니다.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

에이전트가 이전 대화에서 맥락을 어떻게 유지하는지 주목하세요 — 일본, 스시, 사원, 사진 촬영, 3000달러 예산, 혼자 여행, 4월 일정에 대해 알고 있습니다. 짧은 대화에서는 잘 작동하지만, 대화가 길어질수록 전체 기록을 다시 전송하는 비용이 커집니다.

대화가 쌓이는 맥락을 보기 위해 대화를 더 이어가 보겠습니다:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 컨텍스트 요약 패턴

대화가 진행됨에 따라 누적된 컨텍스트를 압축된 형식으로 축약하기 위해 **요약 도구**를 사용할 수 있습니다. 에이전트는 이 도구를 호출하여 주요 선호도를 기록하며, 이전 메시지가 삭제되더라도 필수 정보가 보존되도록 합니다.

이 패턴은 더 정교한 히스토리 축소를 위한 기본 구성 요소입니다:
1. 에이전트가 대화에서 주요 사실을 식별합니다.
2. 요약 도구를 호출하여 이를 저장합니다.
3. 요약이 중요한 내용을 포착하기 때문에 오래된 메시지를 안전하게 제거할 수 있습니다.

아래에서는 에이전트가 배운 내용을 간결하게 요약해 기록할 수 있도록 `summarize_preferences` 도구를 정의합니다.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 요약

이번 강의에서는 Microsoft Agent Framework를 사용하여 장기 에이전트 대화에서 컨텍스트를 관리하는 방법을 배웠습니다:

### 핵심 개념
- **컨텍스트 창은 유한하다** — 대화 내역의 모든 토큰은 비용이 발생하며 한도에 포함됩니다.
- **요약 도구**는 에이전트가 누적된 컨텍스트를 간결한 요약으로 압축하여 토큰 사용량을 줄이면서 필수 정보를 보존할 수 있게 합니다.
- **에이전트 스크래치패드**는 모든 대화 축소 후에도 지속되는 외부 메모리를 제공합니다.

### 여러분이 만든 것
- 여러 턴의 대화에서 연속성을 유지하는 **컨텍스트 인식 에이전트**
- 주요 사용자 세부 정보를 간결한 형식으로 기록하는 **요약 도구**(`summarize_preferences`)
- 컨텍스트 유지 및 변경 처리를 보여주는 **멀티턴 대화**

### 실제 응용
- **고객 서비스 봇**: 긴 지원 세션에서 선호도를 기억
- **개인 비서**: 컨텍스트를 다시 설명하지 않고 진행 중인 프로젝트 추적
- **교육 튜터**: 여러 상호작용에서 학생의 진행 상황 유지

### 다음 단계
- 파일 기반 지속성을 갖춘 완전한 스크래치패드 도구 구현
- 요약 후 자동 히스토리 축소 추가
- 의미 기반 메모리 검색을 위한 벡터 데이터베이스와 결합
- 전체 컨텍스트를 가진 채 며칠 후에 대화를 재개할 수 있는 에이전트 구축


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확성이 포함될 수 있음을 알려드립니다. 원본 문서의 원어본이 권위 있는 출처로 간주되어야 합니다. 중요한 정보의 경우 전문적인 인간 번역을 권장합니다. 본 번역 사용으로 인한 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
